In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
import pyreadr

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
BASE_DIR = Path.cwd().parents[2]

TRAIN_RDS = BASE_DIR / "00_data" / "01_processed" / "03_imputation" / "df_train_imputed.rds"
TEST_RDS  = BASE_DIR / "00_data" / "01_processed" / "03_imputation" / "df_test_imputed.rds"

ANALYSIS_DIR = BASE_DIR / "00_data" / "02_analysis"
OUT_DIR = ANALYSIS_DIR / "tree_models_python_selected_only_fast"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "has_cvd"
RANDOM_STATE = 42

# HPC session resources
AVAILABLE_CORES = 8

# Parallelism strategy:
# - Outer CV/grid search uses all 8 cores
# - Individual models use 1 core each
# This avoids inefficient nested parallelism
CV_N_JOBS = AVAILABLE_CORES
MODEL_N_JOBS = 1
PERM_N_JOBS = AVAILABLE_CORES

# Speed / tuning controls
CV_FOLDS = 3
RF_N_ESTIMATORS = 200
XGB_N_ESTIMATORS = 200
PERM_N_REPEATS = 10

EXCLUDE_VARS = {
    "eid",
    "has_cvd",
    "first_cvd_date",
    "death_date",
}

EXCLUDE_RECRUITMENT_DATE = True
if EXCLUDE_RECRUITMENT_DATE:
    EXCLUDE_VARS.add("recruitment_date")

In [3]:
def find_stability_dir(base_analysis_dir: Path) -> Path:
    candidates = []
    for p in base_analysis_dir.rglob("*"):
        if p.is_dir() and "stability_selection" in p.name:
            candidates.append(p)

    if len(candidates) == 0:
        raise FileNotFoundError(f"No stability-selection folder found inside {base_analysis_dir}")

    preferred = [p for p in candidates if p.name == "stability_selection_sharp_wo_sbp_FINAL"]
    if preferred:
        return preferred[0]

    return sorted(candidates)[0]


def find_selected_txt(stability_dir: Path) -> Path:
    txt_candidates = list(stability_dir.glob("*.txt"))
    if len(txt_candidates) == 0:
        raise FileNotFoundError(f"No TXT files found inside stability-selection folder: {stability_dir}")

    preferred_names = [
        "selected_variable_names.txt",
        "selected_variables_names.txt",
    ]
    for nm in preferred_names:
        p = stability_dir / nm
        if p.exists():
            return p

    selected_like = [p for p in txt_candidates if "selected" in p.name.lower()]
    if selected_like:
        return sorted(selected_like)[0]

    return sorted(txt_candidates)[0]

In [4]:
def read_rds_dataframe(path: Path) -> pd.DataFrame:
    result = pyreadr.read_r(str(path))
    if len(result.keys()) == 0:
        raise ValueError(f"No object found inside {path}")
    df = next(iter(result.values()))
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Object inside {path} is not a DataFrame")
    return df


def load_selected_variables(txt_path: Path) -> list[str]:
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    selected = [x for x in lines if x]
    selected = list(dict.fromkeys(selected))
    return selected


def clean_target(y: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(y):
        return y.astype(int)

    if pd.api.types.is_numeric_dtype(y):
        vals = set(pd.Series(y).dropna().unique().tolist())
        if vals.issubset({0, 1}):
            return y.astype(int)

    y_str = y.astype(str).str.strip().str.lower()
    mapping = {
        "0": 0, "1": 1,
        "false": 0, "true": 1,
        "no": 0, "yes": 1,
    }
    y_mapped = y_str.map(mapping)
    if y_mapped.isna().any():
        bad = y[y_mapped.isna()].unique()
        raise ValueError(f"Target contains values that cannot be mapped to binary 0/1: {bad}")
    return y_mapped.astype(int)


def infer_feature_types(X: pd.DataFrame):
    X = X.copy()

    dt_cols = X.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
    for col in dt_cols:
        X[col] = pd.to_datetime(X[col], errors="coerce")
        X[col] = (X[col] - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)

    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    return X, numeric_cols, categorical_cols


def make_preprocessor(X: pd.DataFrame):
    X2, numeric_cols, categorical_cols = infer_feature_types(X)

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        remainder="drop",
    )

    return X2, preprocessor, numeric_cols, categorical_cols


def evaluate_classifier(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)

    return {
        "auc": auc,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def get_feature_names_from_pipeline(fitted_pipeline, numeric_cols, categorical_cols):
    preprocessor = fitted_pipeline.named_steps["preprocessor"]

    feature_names = []
    if numeric_cols:
        feature_names.extend(numeric_cols)

    if categorical_cols:
        ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
        cat_names = ohe.get_feature_names_out(categorical_cols).tolist()
        feature_names.extend(cat_names)

    return feature_names


def plot_confusion_matrix(y_true, y_pred, title, out_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["no_cvd", "cvd"],
    ).plot(cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_top_importance_bar(df_imp, title, xlabel, out_path, top_n=15):
    plot_df = df_imp.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(plot_df["feature"], plot_df["importance"])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_permutation_boxplot(perm_result, feature_names, title, out_path, top_n=15):
    sorted_idx = np.argsort(perm_result.importances_mean)[-top_n:]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.boxplot(
        perm_result.importances[sorted_idx].T,
        vert=False,
        tick_labels=np.array(feature_names)[sorted_idx],
    )
    ax.set_title(title)
    ax.set_xlabel("Decrease in ROC AUC")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()

In [5]:
SEL_DIR = find_stability_dir(ANALYSIS_DIR)
SEL_TXT = find_selected_txt(SEL_DIR)

print("BASE_DIR:", BASE_DIR)
print("TRAIN_RDS:", TRAIN_RDS)
print("TEST_RDS :", TEST_RDS)
print("SEL_DIR  :", SEL_DIR)
print("SEL_TXT  :", SEL_TXT)

print("TRAIN exists:", TRAIN_RDS.exists())
print("TEST exists :", TEST_RDS.exists())
print("SEL dir exists:", SEL_DIR.exists())
print("SEL txt exists:", SEL_TXT.exists())

if not TRAIN_RDS.exists():
    raise FileNotFoundError(f"Train RDS not found: {TRAIN_RDS}")
if not TEST_RDS.exists():
    raise FileNotFoundError(f"Test RDS not found: {TEST_RDS}")
if not SEL_TXT.exists():
    raise FileNotFoundError(f"Selected-variable TXT file not found: {SEL_TXT}")

BASE_DIR: /rds/general/project/hda_25-26/live/TDS/TDS_Group4
TRAIN_RDS: /rds/general/project/hda_25-26/live/TDS/TDS_Group4/00_data/01_processed/03_imputation/df_train_imputed.rds
TEST_RDS : /rds/general/project/hda_25-26/live/TDS/TDS_Group4/00_data/01_processed/03_imputation/df_test_imputed.rds
SEL_DIR  : /rds/general/project/hda_25-26/live/TDS/TDS_Group4/00_data/02_analysis/stability_selection_runs/stability_selection_sharp_wo_sbp_FINAL
SEL_TXT  : /rds/general/project/hda_25-26/live/TDS/TDS_Group4/00_data/02_analysis/stability_selection_runs/stability_selection_sharp_wo_sbp_FINAL/selected_variable_names.txt
TRAIN exists: True
TEST exists : True
SEL dir exists: True
SEL txt exists: True


In [6]:
print("Loading train/test RDS files...")
df_train = read_rds_dataframe(TRAIN_RDS)
df_test = read_rds_dataframe(TEST_RDS)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

if TARGET not in df_train.columns or TARGET not in df_test.columns:
    raise ValueError(f"Target '{TARGET}' must exist in both train and test data.")

Loading train/test RDS files...


/rds/general/user/gv325/home/anaconda3/envs/TDS_Group_Project/lib/python3.10/site-packages/pyreadr/_pyreadr_parser.py:233: RuntimeWarning: invalid value encountered in cast
  df[colname] = df[colname].values.astype("datetime64[D]").astype(datetime)
/rds/general/user/gv325/home/anaconda3/envs/TDS_Group_Project/lib/python3.10/site-packages/pyreadr/_pyreadr_parser.py:233: RuntimeWarning: invalid value encountered in cast
  df[colname] = df[colname].values.astype("datetime64[D]").astype(datetime)
/rds/general/user/gv325/home/anaconda3/envs/TDS_Group_Project/lib/python3.10/site-packages/pyreadr/_pyreadr_parser.py:233: RuntimeWarning: invalid value encountered in cast
  df[colname] = df[colname].values.astype("datetime64[D]").astype(datetime)
/rds/general/user/gv325/home/anaconda3/envs/TDS_Group_Project/lib/python3.10/site-packages/pyreadr/_pyreadr_parser.py:233: RuntimeWarning: invalid value encountered in cast
  df[colname] = df[colname].values.astype("datetime64[D]").astype(datetime)


Train shape: (197891, 63)
Test shape : (65964, 63)


In [7]:
common_cols = [c for c in df_train.columns if c in df_test.columns]
df_train = df_train[common_cols].copy()
df_test = df_test[common_cols].copy()

y_train = clean_target(df_train[TARGET])
y_test = clean_target(df_test[TARGET])

print("Target cleaned successfully")
print("Train outcome distribution:")
print(y_train.value_counts(dropna=False))
print("Test outcome distribution:")
print(y_test.value_counts(dropna=False))

Target cleaned successfully
Train outcome distribution:
has_cvd
0    176601
1     21290
Name: count, dtype: int64
Test outcome distribution:
has_cvd
0    58816
1     7148
Name: count, dtype: int64


In [8]:
selected_vars = load_selected_variables(SEL_TXT)
print(f"Loaded {len(selected_vars)} selected variables from TXT file.")

selected_predictors = [
    c for c in selected_vars
    if c in df_train.columns and c not in EXCLUDE_VARS
]

if len(selected_predictors) == 0:
    raise ValueError("No selected variables from TXT matched columns in the train/test data.")

print("Number of SELECTED predictors used:", len(selected_predictors))
print("Selected predictors:")
print(selected_predictors)

pd.Series(selected_predictors, name="variable").to_csv(
    OUT_DIR / "selected_predictors_used.csv",
    index=False
)

Loaded 5 selected variables from TXT file.
Number of SELECTED predictors used: 5
Selected predictors:
['triglycerides', 'employment', 'smoking_status_refined', 'housing', 'salt_added']


In [9]:
X_train_sel = df_train[selected_predictors].copy()
X_test_sel = df_test[selected_predictors].copy()

print("X_train_sel shape:", X_train_sel.shape)
print("X_test_sel shape :", X_test_sel.shape)

X_train_sel shape: (197891, 5)
X_test_sel shape : (65964, 5)


In [10]:
rf_out_dir = OUT_DIR / "random_forest"
rf_out_dir.mkdir(parents=True, exist_ok=True)

X_train_rf, rf_preprocessor, rf_numeric_cols, rf_categorical_cols = make_preprocessor(X_train_sel)
X_test_rf, _, _, _ = make_preprocessor(X_test_sel)

rf_pipe = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=MODEL_N_JOBS,
        )),
    ]
)

print("Random Forest pipeline created")
print("Numeric columns:", rf_numeric_cols)
print("Categorical columns:", rf_categorical_cols)

Random Forest pipeline created
Numeric columns: ['triglycerides']
Categorical columns: ['employment', 'smoking_status_refined', 'housing', 'salt_added']


In [11]:
rf_param_grid = {
    "model__max_depth": [None, 12],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt"],
}

rf_grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=CV_FOLDS,
    n_jobs=CV_N_JOBS,
    refit=True,
    verbose=1,
)

n_rf_combos = (
    len(rf_param_grid["model__max_depth"]) *
    len(rf_param_grid["model__min_samples_split"]) *
    len(rf_param_grid["model__min_samples_leaf"]) *
    len(rf_param_grid["model__max_features"])
)

print("Random Forest grid search ready")
print("RF hyperparameter combinations:", n_rf_combos)
print("RF total fits:", n_rf_combos * CV_FOLDS)

Random Forest grid search ready
RF hyperparameter combinations: 8
RF total fits: 24


In [ ]:
rf_grid.fit(X_train_rf, y_train)

rf_best = rf_grid.best_estimator_
print("Random Forest best CV AUC:", rf_grid.best_score_)
print("Random Forest best params:", rf_grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
rf_metrics = evaluate_classifier(rf_best, X_test_rf, y_test)

rf_metrics_table = pd.DataFrame([{
    "model": "Random Forest",
    "cv_folds": CV_FOLDS,
    "n_estimators": RF_N_ESTIMATORS,
    "best_cv_auc": rf_grid.best_score_,
    **rf_grid.best_params_,
    "test_auc": rf_metrics["auc"],
    "test_accuracy": rf_metrics["accuracy"],
    "test_precision": rf_metrics["precision"],
    "test_recall": rf_metrics["recall"],
    "test_f1": rf_metrics["f1"],
}])

rf_metrics_table.to_csv(rf_out_dir / "random_forest_metrics.csv", index=False)

with open(rf_out_dir / "random_forest_best_params.json", "w") as f:
    json.dump(rf_grid.best_params_, f, indent=2)

print(rf_metrics_table)

In [ ]:
plot_confusion_matrix(
    y_test,
    rf_metrics["y_pred"],
    "Random Forest - Confusion Matrix (selected variables only)",
    rf_out_dir / "random_forest_confusion_matrix.png",
)

print("Saved Random Forest confusion matrix")

In [ ]:
rf_feature_names = get_feature_names_from_pipeline(rf_best, rf_numeric_cols, rf_categorical_cols)

rf_native_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_best.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

rf_native_importance.to_csv(rf_out_dir / "random_forest_native_importance.csv", index=False)

plot_top_importance_bar(
    rf_native_importance,
    "Random Forest - Native Importance (selected variables only, Top 15)",
    "Importance",
    rf_out_dir / "random_forest_native_importance_top15.png",
    top_n=15,
)

rf_native_importance.head(15)

In [ ]:
rf_perm_result = permutation_importance(
    estimator=rf_best,
    X=X_test_rf,
    y=y_test,
    scoring="roc_auc",
    n_repeats=PERM_N_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=PERM_N_JOBS,
)

rf_perm_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance_mean": rf_perm_result.importances_mean,
    "importance_std": rf_perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

rf_perm_importance.to_csv(rf_out_dir / "random_forest_permutation_importance.csv", index=False)

plot_permutation_boxplot(
    rf_perm_result,
    rf_feature_names,
    f"Random Forest - Permutation Importance (selected variables only, Top 15, {PERM_N_REPEATS} repeats)",
    rf_out_dir / "random_forest_permutation_importance_top15.png",
    top_n=15,
)

rf_perm_importance.head(15)

In [ ]:
xgb_out_dir = OUT_DIR / "xgboost"
xgb_out_dir.mkdir(parents=True, exist_ok=True)

X_train_xgb, xgb_preprocessor, xgb_numeric_cols, xgb_categorical_cols = make_preprocessor(X_train_sel)
X_test_xgb, _, _, _ = make_preprocessor(X_test_sel)

xgb_pipe = Pipeline(
    steps=[
        ("preprocessor", xgb_preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=RANDOM_STATE,
            n_estimators=XGB_N_ESTIMATORS,
            tree_method="hist",
            n_jobs=MODEL_N_JOBS,
        )),
    ]
)

print("XGBoost pipeline created")
print("Numeric columns:", xgb_numeric_cols)
print("Categorical columns:", xgb_categorical_cols)

In [ ]:
xgb_param_grid = {
    "model__max_depth": [3, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8],
    "model__min_child_weight": [1, 5],
}

xgb_grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=xgb_param_grid,
    scoring="roc_auc",
    cv=CV_FOLDS,
    n_jobs=CV_N_JOBS,
    refit=True,
    verbose=1,
)

n_xgb_combos = (
    len(xgb_param_grid["model__max_depth"]) *
    len(xgb_param_grid["model__learning_rate"]) *
    len(xgb_param_grid["model__subsample"]) *
    len(xgb_param_grid["model__colsample_bytree"]) *
    len(xgb_param_grid["model__min_child_weight"])
)

print("XGBoost grid search ready")
print("XGB hyperparameter combinations:", n_xgb_combos)
print("XGB total fits:", n_xgb_combos * CV_FOLDS)

In [ ]:
xgb_grid.fit(X_train_xgb, y_train)

xgb_best = xgb_grid.best_estimator_
print("XGBoost best CV AUC:", xgb_grid.best_score_)
print("XGBoost best params:", xgb_grid.best_params_)

In [ ]:
xgb_metrics = evaluate_classifier(xgb_best, X_test_xgb, y_test)

xgb_metrics_table = pd.DataFrame([{
    "model": "XGBoost",
    "cv_folds": CV_FOLDS,
    "n_estimators": XGB_N_ESTIMATORS,
    "best_cv_auc": xgb_grid.best_score_,
    **xgb_grid.best_params_,
    "test_auc": xgb_metrics["auc"],
    "test_accuracy": xgb_metrics["accuracy"],
    "test_precision": xgb_metrics["precision"],
    "test_recall": xgb_metrics["recall"],
    "test_f1": xgb_metrics["f1"],
}])

xgb_metrics_table.to_csv(xgb_out_dir / "xgboost_metrics.csv", index=False)

with open(xgb_out_dir / "xgboost_best_params.json", "w") as f:
    json.dump(xgb_grid.best_params_, f, indent=2)

print(xgb_metrics_table)

In [ ]:
plot_confusion_matrix(
    y_test,
    xgb_metrics["y_pred"],
    "XGBoost - Confusion Matrix (selected variables only)",
    xgb_out_dir / "xgboost_confusion_matrix.png",
)

print("Saved XGBoost confusion matrix")

In [ ]:
xgb_feature_names = get_feature_names_from_pipeline(xgb_best, xgb_numeric_cols, xgb_categorical_cols)

xgb_native_importance = pd.DataFrame({
    "feature": xgb_feature_names,
    "importance": xgb_best.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

xgb_native_importance.to_csv(xgb_out_dir / "xgboost_native_importance.csv", index=False)

plot_top_importance_bar(
    xgb_native_importance,
    "XGBoost - Native Importance (selected variables only, Top 15)",
    "Importance",
    xgb_out_dir / "xgboost_native_importance_top15.png",
    top_n=15,
)

xgb_native_importance.head(15)

In [ ]:
xgb_perm_result = permutation_importance(
    estimator=xgb_best,
    X=X_test_xgb,
    y=y_test,
    scoring="roc_auc",
    n_repeats=PERM_N_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=PERM_N_JOBS,
)

xgb_perm_importance = pd.DataFrame({
    "feature": xgb_feature_names,
    "importance_mean": xgb_perm_result.importances_mean,
    "importance_std": xgb_perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

xgb_perm_importance.to_csv(xgb_out_dir / "xgboost_permutation_importance.csv", index=False)

plot_permutation_boxplot(
    xgb_perm_result,
    xgb_feature_names,
    f"XGBoost - Permutation Importance (selected variables only, Top 15, {PERM_N_REPEATS} repeats)",
    xgb_out_dir / "xgboost_permutation_importance_top15.png",
    top_n=15,
)

xgb_perm_importance.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(
    rf_metrics["fpr"],
    rf_metrics["tpr"],
    label=f"RF - selected variables (AUC = {rf_metrics['auc']:.3f})",
)
ax.plot(
    xgb_metrics["fpr"],
    xgb_metrics["tpr"],
    label=f"XGB - selected variables (AUC = {xgb_metrics['auc']:.3f})",
)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_title("ROC comparison - selected variables only")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT_DIR / "combined_roc_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
combined_auc = pd.DataFrame([
    {
        "model": "Random Forest",
        "cv_auc": rf_grid.best_score_,
        "test_auc": rf_metrics["auc"],
        "accuracy": rf_metrics["accuracy"],
        "precision": rf_metrics["precision"],
        "recall": rf_metrics["recall"],
        "f1": rf_metrics["f1"],
    },
    {
        "model": "XGBoost",
        "cv_auc": xgb_grid.best_score_,
        "test_auc": xgb_metrics["auc"],
        "accuracy": xgb_metrics["accuracy"],
        "precision": xgb_metrics["precision"],
        "recall": xgb_metrics["recall"],
        "f1": xgb_metrics["f1"],
    },
])

combined_auc.to_csv(OUT_DIR / "combined_auc_metrics_table.csv", index=False)
combined_auc

In [ ]:
pred_df = pd.DataFrame({
    "y_true": y_test.values,
    "rf_prob": rf_metrics["y_prob"],
    "xgb_prob": xgb_metrics["y_prob"],
    "rf_pred": rf_metrics["y_pred"],
    "xgb_pred": xgb_metrics["y_pred"],
})

pred_df.to_csv(OUT_DIR / "test_predictions.csv", index=False)
pred_df.head()

In [ ]:
run_summary = {
    "base_dir": str(BASE_DIR),
    "analysis_dir": str(ANALYSIS_DIR),
    "stability_dir": str(SEL_DIR),
    "selected_txt": str(SEL_TXT),
    "train_shape": list(df_train.shape),
    "test_shape": list(df_test.shape),
    "n_selected_predictors": len(selected_predictors),
    "selected_predictors": selected_predictors,
    "target": TARGET,
    "excluded_variables": sorted(list(EXCLUDE_VARS)),
    "speed_settings": {
        "available_cores": AVAILABLE_CORES,
        "cv_n_jobs": CV_N_JOBS,
        "model_n_jobs": MODEL_N_JOBS,
        "perm_n_jobs": PERM_N_JOBS,
        "cv_folds": CV_FOLDS,
        "rf_n_estimators": RF_N_ESTIMATORS,
        "xgb_n_estimators": XGB_N_ESTIMATORS,
        "perm_n_repeats": PERM_N_REPEATS,
    },
}

with open(OUT_DIR / "run_summary.json", "w") as f:
    json.dump(run_summary, f, indent=2)

print("Saved run summary")
print("Main output folder:", OUT_DIR)

In [ ]:
print("\n====================================================")
print("Selected-variables-only FAST workflow completed")
print("====================================================")
print("Main output folder:", OUT_DIR)
print("Random Forest folder:", rf_out_dir)
print("XGBoost folder:", xgb_out_dir)
print("====================================================")